# Limpieza de datos

En este notebook:

1. Cargamos las tablas desde `datos/tablas.pkl` (igual que el **paso 4** de `EDA_Jupiter.ipynb`).
2. Aplicamos tratamientos a los valores nulos y erróneos.
3. Guardamos las tablas limpias en `datos/tablas_limpias.pkl`.

**Requisito:** ejecuta antes el paso 2 de `EDA_Jupiter.ipynb` si no existe el fichero de datos.

## 1. Importar librerías

In [2]:
import pickle
from pathlib import Path

import pandas as pd

## 2. Cargar tablas en DataFrames

Replicamos el **paso 4** de `EDA_Jupiter.ipynb`.

In [1]:
from DB import cargar_datos

datos = cargar_datos()

print(f"Tablas cargadas desde fichero: {len(datos)}")
for nombre, df in datos.items():
    print(f"  - {nombre}: {df.shape[0]} filas x {df.shape[1]} columnas")

Tablas cargadas desde fichero: 8
  - cliente: 34 filas x 2 columnas
  - lote: 69567 filas x 2 columnas
  - marca: 35 filas x 2 columnas
  - producto: 70549 filas x 9 columnas
  - proveedor: 35 filas x 2 columnas
  - subtipo: 27 filas x 3 columnas
  - tipo: 15 filas x 2 columnas
  - venta: 70549 filas x 6 columnas


### 2.1 Crear variables para cada tabla

In [3]:
df_cliente = datos["cliente"].copy()
df_lote = datos["lote"].copy()
df_marca = datos["marca"].copy()
df_producto = datos["producto"].copy()
df_proveedor = datos["proveedor"].copy()
df_subtipo = datos["subtipo"].copy()
df_tipo = datos["tipo"].copy()
df_venta = datos["venta"].copy()

### 2.2 Unir venta y producto

La mayoría de limpiezas afectan a **venta** y **producto**. Las unimos para trabajar más fácil.

In [4]:
# Convertir columnas numéricas y fechas
df_venta["precio_venta"] = pd.to_numeric(df_venta["precio_venta"], errors="coerce")
df_venta["tiempo_venta"] = pd.to_datetime(df_venta["tiempo_venta"])
df_producto["coste_inicial"] = pd.to_numeric(df_producto["coste_inicial"], errors="coerce")
df_producto["tiempo_recogida"] = pd.to_datetime(df_producto["tiempo_recogida"])

# Tabla combinada (una fila = una venta con su producto)
df = df_venta.merge(df_producto, on="id_producto")

print(f"Registros combinados: {len(df)}")

Registros combinados: 70549


## 3. Diagnóstico antes de limpiar

Vemos cuántos problemas hay en los datos.

In [5]:
columnas_revisar = ["coste_inicial", "precio_venta", "tiempo_venta"]

print("Valores nulos:")
print(df[columnas_revisar].isnull().sum())

print(f"\nPeso negativo: {(df['peso'] < 0).sum()}")

mas_de_un_nulo = (df[columnas_revisar].isnull().sum(axis=1) > 1).sum()
print(f"Registros con más de un nulo: {mas_de_un_nulo}")

Valores nulos:
coste_inicial    2011
precio_venta      694
tiempo_venta      223
dtype: int64

Peso negativo: 113
Registros con más de un nulo: 27


---
## 4. Tratamiento 1 — Eliminar registros con más de un nulo

Si un registro tiene **más de un nulo** entre `coste_inicial`, `precio_venta` y `tiempo_venta`, lo eliminamos.

In [6]:
antes = len(df)

df["n_nulos"] = df[columnas_revisar].isnull().sum(axis=1)
df = df[df["n_nulos"] <= 1].copy()
df = df.drop(columns="n_nulos")

print(f"Registros eliminados: {antes - len(df)}")
print(f"Registros restantes: {len(df)}")

Registros eliminados: 27
Registros restantes: 70522


## 5. Tratamiento 2 — Rellenar `coste_inicial` nulos

Usamos la **media del coste** del mismo **tipo de fruta** en una ventana de **3 días** centrada en el día de recogida (día anterior, día de recogida y día siguiente).

In [7]:
def rellenar_media_3_dias(df, columna, columna_fecha, columna_tipo):
    """Rellena nulos con la media de 3 días (d-1, d, d+1) por tipo de fruta."""
    df = df.copy()
    df["_fecha"] = pd.to_datetime(df[columna_fecha]).dt.normalize()

    # Medias diarias solo con valores válidos
    medias_dia = (
        df[df[columna].notna()]
        .groupby([columna_tipo, "_fecha"])[columna]
        .mean()
        .reset_index()
    )

    def calcular_media(fila):
        tipo = fila[columna_tipo]
        fecha = fila["_fecha"]
        fechas = [fecha - pd.Timedelta(days=1), fecha, fecha + pd.Timedelta(days=1)]

        valores = medias_dia[
            (medias_dia[columna_tipo] == tipo) & (medias_dia["_fecha"].isin(fechas))
        ][columna]

        if len(valores) == 0:
            return medias_dia[medias_dia[columna_tipo] == tipo][columna].mean()
        return valores.mean()

    nulos = df[columna].isna()
    print(f"  Nulos a rellenar en {columna}: {nulos.sum()}")

    if nulos.any():
        df.loc[nulos, columna] = df.loc[nulos].apply(calcular_media, axis=1)

    return df.drop(columns="_fecha")


df = rellenar_media_3_dias(df, "coste_inicial", "tiempo_recogida", "id_tipo")
print(f"Coste_inicial nulos restantes: {df['coste_inicial'].isnull().sum()}")

  Nulos a rellenar en coste_inicial: 1986
Coste_inicial nulos restantes: 0


## 6. Tratamiento 3 — Rellenar `precio_venta` nulos

Misma idea: media de **3 días centrados en el día de venta**, por **tipo de fruta**.

In [8]:
df = rellenar_media_3_dias(df, "precio_venta", "tiempo_venta", "id_tipo")
print(f"Precio_venta nulos restantes: {df['precio_venta'].isnull().sum()}")

  Nulos a rellenar en precio_venta: 670
Precio_venta nulos restantes: 0


## 7. Tratamiento 4 — Rellenar `tiempo_venta` nulos

Si falta la fecha de venta, usamos el **día siguiente** a la fecha de recogida.

In [9]:
sin_fecha = df["tiempo_venta"].isna()
print(f"Tiempo_venta nulos a rellenar: {sin_fecha.sum()}")

df.loc[sin_fecha, "tiempo_venta"] = df.loc[sin_fecha, "tiempo_recogida"] + pd.Timedelta(days=1)

print(f"Tiempo_venta nulos restantes: {df['tiempo_venta'].isnull().sum()}")

Tiempo_venta nulos a rellenar: 218
Tiempo_venta nulos restantes: 0


## 8. Tratamiento 5 — Eliminar registros con peso negativo

In [10]:
antes = len(df)

df = df[df["peso"] >= 0].copy()

print(f"Registros eliminados (peso negativo): {antes - len(df)}")
print(f"Registros restantes: {len(df)}")

Registros eliminados (peso negativo): 113
Registros restantes: 70409


## 9. Separar de nuevo en tablas `venta` y `producto`

In [11]:
columnas_venta = ["id_venta", "id_producto", "id_cliente", "precio_venta", "tiempo_venta", "peso"]
columnas_producto = [c for c in df_producto.columns if c in df.columns]

df_venta = df[columnas_venta].copy()
df_producto = df[columnas_producto].drop_duplicates(subset="id_producto").copy()

print(f"df_venta: {df_venta.shape}")
print(f"df_producto: {df_producto.shape}")

df_venta: (70409, 6)
df_producto: (70409, 9)


## 10. Comprobar resultado final

In [12]:
print("=== VENTA ===")
print("Nulos:")
print(df_venta.isnull().sum())
print(f"Peso negativo: {(df_venta['peso'] < 0).sum()}")

print("\n=== PRODUCTO ===")
print("Nulos:")
print(df_producto.isnull().sum())

=== VENTA ===
Nulos:
id_venta        0
id_producto     0
id_cliente      0
precio_venta    0
tiempo_venta    0
peso            0
dtype: int64
Peso negativo: 0

=== PRODUCTO ===
Nulos:
id_producto        0
t_id               0
id_tipo            0
id_subtipo         0
id_marca           0
id_proveedor       0
id_lote            0
coste_inicial      0
tiempo_recogida    0
dtype: int64


## 11. Guardar tablas limpias

Guardamos todas las tablas (las limpias y el resto sin cambios) para usarlas en otros notebooks, como `Queries en Python_Despues.ipynb`.

In [13]:
datos_limpios = {
    "cliente": df_cliente,
    "lote": df_lote,
    "marca": df_marca,
    "producto": df_producto,
    "proveedor": df_proveedor,
    "subtipo": df_subtipo,
    "tipo": df_tipo,
    "venta": df_venta,
}

ruta_limpias = Path("datos") / "tablas_limpias.pkl"
ruta_limpias.parent.mkdir(parents=True, exist_ok=True)

with open(ruta_limpias, "wb") as f:
    pickle.dump(datos_limpios, f)

print(f"Tablas limpias guardadas en: {ruta_limpias}")

Tablas limpias guardadas en: datos\tablas_limpias.pkl
